In [1]:
%pip install pyrosm geopandas -q


StatementMeta(, 89b51ab6-c564-45ed-887b-5d8babce47a1, 7, Finished, Available, Finished, True)

databricks-sdk 0.120.0 requires protobuf!=5.26.*,!=5.27.*,!=5.28.*,!=5.29.0,!=5.29.1,!=5.29.2,!=5.29.3,!=5.29.4,!=6.30.0,!=6.30.1,!=6.31.0,<7.0,>=4.25.8, but you have protobuf 7.36.2 which is incompatible.
mlflow-skinny 2.22.0 requires protobuf<7,>=3.12.0, but you have protobuf 7.36.2 which is incompatible.

Note: you may need to restart the kernel to use updated packages.



In [2]:
import geopandas as gpd, requests, os, shutil
from pyrosm import OSM

# ---------- Paramètres ----------
URL_OSM = "https://download.geofabrik.de/europe/france/provence-alpes-cote-d-azur-latest.osm.pbf"
BASE = "/lakehouse/default/Files/osm"
BRUT = f"{BASE}/brut"
os.makedirs(BRUT, exist_ok=True)

# ---------- 1. Télécharger OpenStreetMap PACA (même source que le fichier du PC, ~380 Mo) ----------
pbf = f"{BRUT}/provence-alpes-cote-d-azur-latest.osm.pbf"
if not os.path.exists(pbf):
    with requests.get(URL_OSM, stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(pbf, "wb") as f:
            for morceau in r.iter_content(1024 * 1024):
                f.write(morceau)
print("Fichier OSM :", round(os.path.getsize(pbf) / 1e6), "MB")
shutil.copy(pbf, "/tmp/paca.osm.pbf")          # copie locale = lecture plus rapide

# ---------- 2. Zones : Toulon Métropole et Marseille + 1 km de marge ----------
tpm = gpd.read_file("https://geo.api.gouv.fr/epcis/248300543/communes?fields=nom,code,contour&format=geojson&geometry=contour")
mrs = gpd.read_file("https://geo.api.gouv.fr/communes?code=13055&fields=nom,code,contour&format=geojson&geometry=contour")
zones = {"toulon": tpm, "marseille": mrs}

def sauver(gdf, chemin):
    if gdf is None or len(gdf) == 0:
        print("   vide :", os.path.basename(chemin)); return
    gdf = gdf.copy()
    for c in gdf.columns:
        if c != "geometry" and gdf[c].dtype == object:
            gdf[c] = gdf[c].astype(str)
    gdf.to_parquet(chemin)
    print(f"   {os.path.basename(chemin):28s} {len(gdf):>8,} lignes")

# ---------- 3. Extraire pour chaque ville ----------
for ville, communes in zones.items():
    print("\n=====", ville, "=====")
    zone = gpd.GeoSeries([communes.to_crs(2154).union_all().buffer(1000)], crs=2154).to_crs(4326).iloc[0]
    osm = OSM("/tmp/paca.osm.pbf", bounding_box=zone)
    dossier = f"{BASE}/{ville}"
    os.makedirs(dossier, exist_ok=True)

    noeuds, troncons = osm.get_network(network_type="walking", nodes=True)     # rues et chemins piétons
    sauver(troncons, f"{dossier}/pieton_troncons.parquet")
    sauver(noeuds,   f"{dossier}/pieton_noeuds.parquet")

    sauver(osm.get_pois(custom_filter={"amenity": ["parking"]}), f"{dossier}/parkings.parquet")
    sauver(osm.get_pois(custom_filter={"shop": True, "office": True,
           "amenity": ["school", "college", "university", "hospital", "clinic", "townhall", "marketplace", "library"]}),
           f"{dossier}/activites.parquet")
    sauver(osm.get_pois(custom_filter={"amenity": ["car_sharing", "bicycle_rental"]}), f"{dossier}/partage_existant.parquet")

print("\nTerminé")

StatementMeta(, 89b51ab6-c564-45ed-887b-5d8babce47a1, 9, Finished, Available, Finished, False)

Fichier OSM : 391 MB

===== toulon =====


Reading paca.osm.pbf:  42%|####1     | 164M/391M [00:02<00:02, 81.4MB/s]

   pieton_troncons.parquet       278,553 lignes
   pieton_noeuds.parquet         262,663 lignes


Reading paca.osm.pbf:  34%|###3      | 131M/391M [00:02<00:03, 65.2MB/s]

   parkings.parquet                2,487 lignes


... (17 outputs left out)

StatementMeta(, 89b51ab6-c564-45ed-887b-5d8babce47a1, 12, Finished, Available, Finished, False)

No such comm: 862ad1fe084049c3ba0eeef8e0ae8fdc
No such comm: 64112ba2c30b4096a1a49970c391573b


StatementMeta(, 89b51ab6-c564-45ed-887b-5d8babce47a1, 13, Finished, Available, Finished, False)

No such comm: 9fbf1c01e3254b549255bff63b96b792


In [3]:
import geopandas as gpd
p = gpd.read_parquet("/lakehouse/default/Files/osm/toulon/partage_existant.parquet")
cols = [c for c in ["amenity", "name", "operator", "network", "brand"] if c in p.columns]
display(p[cols])

StatementMeta(, 89b51ab6-c564-45ed-887b-5d8babce47a1, 14, Finished, Available, Finished, False)